# GRPO (RLAIF) on Colab Pro (L4 preferred, T4 fallback)

One command per docs/DESIGN.md: clone → install → secrets → `scripts/grpo_colab.py`.

That bootstrap downloads the tokenizer + pref split (rollout prompts) from the Hub,
then runs the GRPO stage (ts2-grpo with --resume) — which enforces issue 05's accuracy
gate, initializes the policy + frozen reference from the SFT checkpoint via `[init].hub_source`,
loads the frozen Reward Model via `[reward].hub_source`, samples G rollouts per Slot Prompt,
and optimizes the policy with the group-relative PPO loss + KL leash, checkpointing back to the
Hub. It streams reward / KL / Self-BLEU to W&B so reward hacking and diversity collapse are
visible early. It is idempotent: after an L4 preemption, just re-run the last cell to resume
from the latest Hub checkpoint. All logic lives in the package/script; edit
`configs/grpo_full.toml`, not here. See `docs/colab-notes.md` for the run procedure.

Before running: set `HF_TOKEN` and `WANDB_API_KEY` in Colab **Secrets** (key icon, left sidebar),
set the repo URL below, and on a T4 change `precision = "fp16"` in the config (Turing has no bf16).

In [ ]:
REPO_URL = "https://github.com/harryct229/tinystories_v2.git"
!git clone {REPO_URL}
%cd tinystories_v2
!pip install -q -e '.[track]'

In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

In [ ]:
!python scripts/grpo_colab.py